# Crop Classification Using Satellite Data
# Area: Mahanadi River Delta, Odisha
# Classes: Non-vegetation (Sand), Sparse vegetation, Dense vegetation

In [29]:
!pip install earthengine-api geemap -q

import ee
import geemap
import pandas as pd
import matplotlib.pyplot as plt

ee.Authenticate()
ee.Initialize(project='gen-lang-client-0869531784')

aoi = ee.Geometry.Rectangle([86.00, 20.20, 86.15, 20.35])
Map = geemap.Map(center=[20.275, 86.075], zoom=12)
Map.addLayer(aoi, {}, "Area of Interest")
Map


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Map(center=[20.275, 86.075], controls=(WidgetControl(options=['position', 'transparent_bg'], position='toprigh…

In [30]:
start_date = "2024-11-01"
end_date = "2025-02-28"

s2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(aoi)
    .filterDate(start_date, end_date)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
)

image = s2.median().clip(aoi)

bands = ["B2", "B3", "B4", "B8", "B11", "B12"]

In [31]:
# Compute NDVI
# NDVI = (NIR - Red) / (NIR + Red)
# Sentinel-2: NIR = B8, Red = B4

ndvi = image.normalizedDifference(["B8", "B4"]).rename("NDVI")
image_with_ndvi = image.select(bands).addBands(ndvi)

rgb_vis = {
    "bands": ["B4", "B3", "B2"],
    "min": 0,
    "max": 3000
}

ndvi_vis = {
    "min": -0.2,
    "max": 0.8,
    "palette": ["brown", "yellow", "green"]
}

Map = geemap.Map(center=[20.275, 86.075], zoom=12)
Map.addLayer(image, rgb_vis, "Sentinel-2 RGB")
Map.addLayer(ndvi, ndvi_vis, "NDVI")
Map.addLayer(aoi, {}, "AOI")
Map

Map(center=[20.275, 86.075], controls=(WidgetControl(options=['position', 'transparent_bg'], position='toprigh…

In [32]:
# Class 0: Non-vegetation (Sand)
non_veg = ee.FeatureCollection([
    ee.Feature(
        ee.Geometry.Polygon([[
            [86.0538, 20.3256],
            [86.0558, 20.3256],
            [86.0558, 20.3276],
            [86.0538, 20.3276],
            [86.0538, 20.3256]
        ]]),
        {"class": 0}
    )
])

# Class 1: Sparse vegetation
sparse_veg = ee.FeatureCollection([
    ee.Feature(
        ee.Geometry.Polygon([[
            [86.120, 20.260],
            [86.125, 20.260],
            [86.125, 20.265],
            [86.120, 20.265],
            [86.120, 20.260]
        ]]),
        {"class": 1}
    )
])

# Class 2: Dense vegetation / crop
dense_veg = ee.FeatureCollection([
    ee.Feature(
        ee.Geometry.Polygon([[
            [86.080, 20.230],
            [86.085, 20.230],
            [86.085, 20.235],
            [86.080, 20.235],
            [86.080, 20.230]
        ]]),
        {"class": 2}
    )
])

training_polygons = non_veg.merge(sparse_veg).merge(dense_veg)

Map = geemap.Map(center=[20.275, 86.075], zoom=12)
Map.addLayer(image, rgb_vis, "Sentinel-2 RGB")
Map.addLayer(training_polygons, {}, "Training Samples")
Map

Map(center=[20.275, 86.075], controls=(WidgetControl(options=['position', 'transparent_bg'], position='toprigh…

In [33]:
input_bands = bands + ["NDVI"]

samples = image_with_ndvi.select(input_bands).sampleRegions(
    collection=training_polygons,
    properties=["class"],
    scale=10
)
#70% training, 30% testing
samples = samples.randomColumn("random")
train_samples = samples.filter(ee.Filter.lt("random", 0.7))
test_samples = samples.filter(ee.Filter.gte("random", 0.7))

print("Training samples:", train_samples.size().getInfo())
print("Testing samples:", test_samples.size().getInfo())

Training samples: 4739
Testing samples: 2017


In [34]:
# Train Random Forest Classifier

classifier = ee.Classifier.smileRandomForest(numberOfTrees=50).train(
    features=train_samples,
    classProperty="class",
    inputProperties=input_bands
)

classified = image_with_ndvi.select(input_bands).classify(classifier)

class_vis = {
    "min": 0,
    "max": 2,
    "palette": ["gray", "yellow", "green"]
}

Map = geemap.Map(center=[20.275, 86.075], zoom=12)
Map.addLayer(image, rgb_vis, "Sentinel-2 RGB")
Map.addLayer(ndvi, ndvi_vis, "NDVI")
Map.addLayer(classified, class_vis, "Classification Map")
Map.addLayer(aoi, {}, "AOI")
Map

Map(center=[20.275, 86.075], controls=(WidgetControl(options=['position', 'transparent_bg'], position='toprigh…

In [35]:
# CONFUSION MATRIX

validated = test_samples.classify(classifier)
confusion_matrix = validated.errorMatrix("class", "classification")

matrix_list = confusion_matrix.getInfo()
accuracy = confusion_matrix.accuracy().getInfo()
kappa = confusion_matrix.kappa().getInfo()

df_cm = pd.DataFrame(
    matrix_list,
    index=["Non-vegetation (Sand)", "Sparse Vegetation", "Dense Vegetation"],
    columns=["Predicted Non-veg", "Predicted Sparse", "Predicted Dense"]
)

print("=== CLASSIFICATION METRICS ===")
print(f"Overall Accuracy : {round(accuracy * 100, 2)}%")
print(f"Kappa Score      : {round(kappa, 4)}\n")
print("=== CONFUSION MATRIX ===")
print(df_cm)

=== CLASSIFICATION METRICS ===
Overall Accuracy : 94.15%
Kappa Score      : 0.8967

=== CONFUSION MATRIX ===
                       Predicted Non-veg  Predicted Sparse  Predicted Dense
Non-vegetation (Sand)                147                 0                1
Sparse Vegetation                      2               893               52
Dense Vegetation                       6                57              859


In [36]:
# Export Output Maps to Google Drive

task1 = ee.batch.Export.image.toDrive(
    image=ndvi,
    description="NDVI_Map",
    folder="Crop_Classification_Output",
    fileNamePrefix="ndvi_map",
    region=aoi,
    scale=10,
    maxPixels=1e13
)

task2 = ee.batch.Export.image.toDrive(
    image=classified,
    description="Classification_Map",
    folder="Crop_Classification_Output",
    fileNamePrefix="classification_map",
    region=aoi,
    scale=10,
    maxPixels=1e13
)

task1.start()
task2.start()

print("Export started. Confirm the tasks in the GEE Code Editor to finalize.")

Export started. Confirm the tasks in the GEE Code Editor to finalize.
